# Parameter Scan Analysis and Comparison Visualisation

Loads completed scan results from `runs/`, builds a summary DataFrame,
and generates the full suite of comparison visualisations:

| § | Plot | Function |
|---|------|----------|
| 3 | η vs pressure (line) | `plot_scan_eta_vs_pressure` |
| 4 | K_eff vs pressure (line) | `plot_scan_keff_vs_pressure` |
| 5 | Method comparison bar chart | `plot_scan_method_comparison_bar` |
| 6 | K_eff × pressure heatmap | `plot_scan_heatmap` |
| 7 | Small-multiple η(t) grid | `plot_scan_neutralization_timeseries_grid` |
| 8 | Final η grouped by gas | `plot_scan_final_eta_bar_by_gas` |
| 9 | Bunched-beam K_eff,peak | `plot_bunched_beam_keff` (multi-case) |

> **Note**: Cells work on any completed subset — no results means synthetic
> fallback data is used so the notebook always renders end-to-end.


In [ ]:
from pathlib import Path
import os, sys, subprocess, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate project root
_ROOT = Path.cwd()
while _ROOT.name != 'plasma_column' and _ROOT.parent != _ROOT:
    _ROOT = _ROOT.parent
if str(_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_ROOT / 'src'))

WORK           = Path.home() / 'Work' / 'simulation_codes-working'
WARPX_DATA_DIR = WORK / 'warpx-data'
RUNS_DIR       = _ROOT / 'runs'
PLOTS_DIR      = _ROOT / 'plots'
RUNS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

os.environ['WARPX_DATA_DIR']  = str(WARPX_DATA_DIR)
os.environ['LD_LIBRARY_PATH'] = (
    str(WORK / 'warpx' / 'install' / 'lib') + ':'
    + os.environ.get('LD_LIBRARY_PATH', '')
)
print('Python :', sys.executable)
print('ROOT   :', _ROOT)
print('WarpX data:', WARPX_DATA_DIR)


In [ ]:
from plasma_column.notebook_utils import print_simulation_config
_DEFAULTS = {
    'runs root':               str(RUNS_DIR),
    'figures root':            str(PLOTS_DIR),
    'scan YAML (pressure)':    'cases/pressure_scan_h2_kr.yaml',
    'scan YAML (method)':      'cases/method_scan_baseline.yaml',
    't_avg window (frac)':     0.25,
    'beam energy [keV]':       30.0,
    'bunching factors':        '1, 2, 3, 5, 8',
}
print_simulation_config(
    notebook_title='Parameter Scan Analysis',
    defaults=_DEFAULTS, overrides={},
)


## 1. Import scan functions


In [ ]:
import warnings
from plasma_column.run_matrix import (
    ScanMatrix, ScanParameter,
    build_scan_dataframe,
    collect_scan_results,
    save_scan_summary,
    load_scan_summary,
)
from plasma_column.plotting import (
    setup_publication_style,
    plot_scan_eta_vs_pressure,
    plot_scan_keff_vs_pressure,
    plot_scan_method_comparison_bar,
    plot_scan_heatmap,
    plot_scan_neutralization_timeseries_grid,
    plot_scan_final_eta_bar_by_gas,
    plot_bunched_beam_keff,
    plot_multi_case_neutralization,
)
from plasma_column.diagnostics import (
    load_particle_number_diagnostic,
    compute_particle_number_metrics,
)
setup_publication_style()
print('Scan analysis helpers loaded.')


## 2. Build scan index and collect results

Constructs the expected scan DataFrame from the YAML configs, then
calls `collect_scan_results()` to load whatever has finished.
Missing cases show NaN; already-complete cases are loaded automatically.


In [ ]:
PRESSURES = [1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 3e-4]

# Pressure scan index (seeded)
_scan_s = ScanMatrix(
    scan_name='pressure_scan', script=_ROOT/'plasma_column_mcc_picmi_v7.py',
    parameters=[ScanParameter('pressure_torr', PRESSURES)],
    gases=['H2','Kr'], methods=['seeded'], runs_root=RUNS_DIR,
)
# Pressure scan index (callback)
_scan_c = ScanMatrix(
    scan_name='callback_scan', script=_ROOT/'plasma_column_callback_source_picmi_v3.py',
    parameters=[ScanParameter('pressure_torr', PRESSURES)],
    gases=['H2','Kr'], methods=['callback'], runs_root=RUNS_DIR,
)

df_index = pd.concat([
    build_scan_dataframe(_scan_s),
    build_scan_dataframe(_scan_c),
], ignore_index=True)

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    summary = collect_scan_results(df_index, RUNS_DIR)

ok = summary[summary['status']=='ok']
print(f'Cases found: {len(ok)}/{len(summary)}')
summary[['case_name','gas','method','pressure_torr',
         'final_eta_net','final_keff_over_k0','status']]


In [ ]:
# Save summary CSV for later use
_csv = RUNS_DIR.parent / 'plots' / 'scan_summary.csv'
if len(ok) > 0:
    save_scan_summary(summary, _csv)
    print('Saved summary →', _csv)

# ── Synthetic fallback when no runs exist yet ─────────────────────────────
# This ensures every plot cell below renders even before simulations finish.
if len(ok) == 0:
    print('No completed cases — using SYNTHETIC fallback data.')
    _rows = []
    for gas, eta_scale in [('H2',0.60),('Kr',0.87)]:
        for method, boost in [('seeded',1.0),('callback',1.15)]:
            for p in PRESSURES:
                eta = float(np.clip(eta_scale * boost * (p/1e-5)**0.45, 0, 0.99))
                _rows.append({'case_name':f'{method}_{gas}_{p:.0e}',
                               'gas':gas,'method':method,'pressure_torr':p,
                               'final_eta_net':eta,'final_eta_electron_only':eta*0.95,
                               'final_keff_over_k0':1-eta,'avg_eta_net':eta*0.9,
                               'status':'synthetic'})
    summary = pd.DataFrame(_rows)
    ok = summary
    print(f'Synthetic summary built ({len(summary)} rows)')


## 3. η_net vs pressure — line plot

Primary result: how fast neutralisation saturates as pressure increases.
H₂ and Kr are expected to converge toward η=1 at high pressure but with
different rates due to their cross-section magnitudes.


In [ ]:
p, _ = plot_scan_eta_vs_pressure(
    summary, PLOTS_DIR,
    eta_col='final_eta_net',
    title=r'Final $\eta_{\rm net}$ vs Gas Pressure — H$_2$ and Kr',
    output_name='scan_eta_vs_pressure',
)
plt.show()
print('Saved:', p.name)


## 4. K_eff/K0 vs pressure — line plot

Companion to §3: shows the perveance reduction directly.
The target is K_eff/K0 < 0.5 (>50% reduction) at the operating pressure.


In [ ]:
p, _ = plot_scan_keff_vs_pressure(
    summary, PLOTS_DIR,
    keff_col='final_keff_over_k0',
    title=r'$K_{\rm eff}/K_0$ vs Pressure — Seeded and Callback Methods',
    output_name='scan_keff_vs_pressure',
)
plt.show()
print('Saved:', p.name)


## 5. Method comparison bar chart

Side-by-side final K_eff/K0 for every case at a glance.
Bars coloured by gas; sorted by case_name.


In [ ]:
p, _ = plot_scan_method_comparison_bar(
    summary.sort_values(['method','gas','pressure_torr']),
    PLOTS_DIR,
    metric_col='final_keff_over_k0',
    ylabel=r'$K_{\rm eff}/K_0$',
    title='Method and Pressure Comparison — Final Effective Perveance',
    output_name='scan_method_comparison_bar',
)
plt.show()
print('Saved:', p.name)


## 6. Parameter scan heatmap: gas × pressure

2-D colour map: rows = gas, columns = pressure. Cell colour and value
show K_eff/K0. Green = well-compensated; red = poor.
Use `row_col='method'` to pivot by method instead.


In [ ]:
for method in summary['method'].unique():
    sub = summary[summary['method']==method].copy()
    if len(sub) < 2: continue
    p, _ = plot_scan_heatmap(
        sub, PLOTS_DIR,
        row_col='gas', col_col='pressure_torr',
        value_col='final_keff_over_k0',
        title=fr'$K_{{\rm eff}}/K_0$ Heatmap — {method} method',
        output_name=f'scan_heatmap_{method}',
    )
    plt.show()
    print('Saved:', p.name)


## 7. Small-multiple η(t) grid — all scan cases

One mini-panel per case shows the neutralisation build-up trajectory.
All panels share the y-axis (0–1) for easy comparison.
Cases without diagnostic data show empty panels.


In [ ]:
# Load time-series for all completed cases
ts_pairs = []
for _, row in summary.iterrows():
    out_d = RUNS_DIR / row['case_name']
    from plasma_column.run_matrix import _find_diag
    diag = _find_diag(out_d)
    if diag is None:
        # Synthetic time-series for demonstration
        _t = np.linspace(0, 400e-9, 150)
        eta_f = row.get('final_eta_net', 0.5)
        _eta  = eta_f * (1 - np.exp(-_t / (120e-9)))
        _df = pd.DataFrame({'time': _t, 'eta_net': _eta})
    else:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            _df = compute_particle_number_metrics(
                load_particle_number_diagnostic(diag))
    label = f"{row['gas']} | {row['method']} | {row['pressure_torr']:.1e} Torr"
    ts_pairs.append((label, _df))

if ts_pairs:
    p, _ = plot_scan_neutralization_timeseries_grid(
        ts_pairs, PLOTS_DIR, ncols=4,
        output_name='scan_timeseries_grid',
        title='Neutralisation Build-up — All Scan Cases',
    )
    plt.show()
    print('Saved:', p.name)


## 8. Final η grouped by gas and method (bar)

Clusters show the gas comparison within each method.
Use `group_col='gas'` to flip and cluster by gas instead.


In [ ]:
p, _ = plot_scan_final_eta_bar_by_gas(
    summary, PLOTS_DIR,
    eta_col='final_eta_net',
    group_col='method',
    title='Final Neutralisation — Grouped by Method and Gas',
    output_name='scan_final_eta_bar',
)
plt.show()
print('Saved:', p.name)


## 9. Bunched-beam K_eff,peak at best operating point

Selects the case with the highest η_net and plots
K_eff,peak/K0 for B_f = 1…8 as the final quantitative result.

**Interpretation**: even at η̄ = 0.9, a bunching factor of 5 leaves
K_eff,peak/K0 ≈ 0.82 — a significant residual space-charge load.


In [ ]:
# Select best case by lowest final_keff_over_k0
best_row = summary.loc[summary['final_keff_over_k0'].idxmin()]
best_name = best_row['case_name']
print(f'Best case: {best_name}  K_eff/K0 = {best_row["final_keff_over_k0"]:.4f}')

# Load its time-series (or use synthetic)
_d = RUNS_DIR / best_name
from plasma_column.run_matrix import _find_diag
_diag = _find_diag(_d)
if _diag is not None:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        _best_hist = compute_particle_number_metrics(
            load_particle_number_diagnostic(_diag))
    _t_ns    = _best_hist['time'].values * 1e9
    _eta_avg = _best_hist['eta_net'].values.clip(0, 1)
else:
    _t_ns    = np.linspace(0, 400, 200)
    eta_f    = float(best_row['final_eta_net'])
    _eta_avg = eta_f * (1 - np.exp(-_t_ns / 120))
    print('(using synthetic time-series for best case)')

p, _ = plot_bunched_beam_keff(
    _t_ns, _eta_avg, PLOTS_DIR,
    case_name=f'best_case_{best_name}',
    bunching_factors=[1.0, 2.0, 3.0, 5.0, 8.0],
)
plt.show()
print('Saved:', p.name)


## 10. Summary table


In [ ]:
display(
    summary[['case_name','gas','method','pressure_torr',
              'final_eta_electron_only','final_eta_net',
              'final_keff_over_k0','avg_eta_net','n_steps','status']]
    .sort_values(['method','gas','pressure_torr'])
    .style.format({
        'pressure_torr':           '{:.2e}',
        'final_eta_electron_only': '{:.4f}',
        'final_eta_net':           '{:.4f}',
        'final_keff_over_k0':      '{:.4f}',
        'avg_eta_net':             '{:.4f}',
    })
)
